In [0]:
catalog="Ecommerce"

###***Products***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

gld_brands=spark.read.table(f"{catalog}.silver.slv_brands")
gld_products=spark.read.table(f"{catalog}.silver.slv_products")
gld_categories=spark.read.table(f"{catalog}.silver.slv_category")

gld_brands.createOrReplaceTempView("brands")
gld_products.createOrReplaceTempView("products")
gld_categories.createOrReplaceTempView("category")

spark.sql(f"USE CATALOG {catalog}")


In [0]:
%sql
Create or replace table Ecommerce.Gold.Gld_products_dim as
with brand_category as (
  select
    c.category_code,
    c.category_name,
    b.brand_code,
    b.brand_name
  from
    category c
      join brands b
        on c.category_code = b.category_code
)
select
  p.product_id,
  p.sku,
  p.category_code,
  COALESCE(bc.category_name, 'Not Available') AS category_name,
  p.brand_code,
  COALESCE(bc.brand_name, 'Not Available') AS brand_name,
  p.color,
  p.size,
  p.material,
  p.weight_grams,
  p.length_cm,
  p.width_cm,
  p.height_cm,
  p.rating_count,
  p.Source_file,
  p.Ingested_at
from
  products p
    Left join brand_category bc
      on p.brand_code = bc.brand_code

###***Customers***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

gld_customers=spark.read.table(f"{catalog}.silver.slv_customers")


# India states
india_region = {
    "MH": "West", "GJ": "West", "RJ": "West",
    "KA": "South", "TN": "South", "TS": "South", "AP": "South", "KL": "South",
    "UP": "North", "WB": "North", "DL": "North"
}
# Australia states
australia_region = {
    "VIC": "SouthEast", "WA": "West", "NSW": "East", "QLD": "NorthEast"
}

# United Kingdom states
uk_region = {
    "ENG": "England", "WLS": "Wales", "NIR": "Northern Ireland", "SCT": "Scotland"
}

# United States states
us_region = {
    "MA": "NorthEast", "FL": "South", "NJ": "NorthEast", "CA": "West", 
    "NY": "NorthEast", "TX": "South"
}

# UAE states
uae_region = {
    "AUH": "Abu Dhabi", "DU": "Dubai", "SHJ": "Sharjah"
}

# Singapore states
singapore_region = {
    "SG": "Singapore"
}

# Canada states
canada_region = {
    "BC": "West", "AB": "West", "ON": "East", "QC": "East", "NS": "East", "IL": "Other"
}

# Combine into a master dictionary
country_state_map = {
    "India": india_region,
    "Australia": australia_region,
    "United Kingdom": uk_region,
    "United States": us_region,
    "United Arab Emirates": uae_region,
    "Singapore": singapore_region,
    "Canada": canada_region
}  

rows = []
for country, states in country_state_map.items():
    for state_code, region in states.items():
        rows.append(Row(country=country, state=state_code, region=region))

df_region=spark.createDataFrame(rows)

gld_customers=gld_customers.join(df_region, on=['country','state'], how="left")
gld_customers=gld_customers.fillna('other', subset=['region'])

gld_customers.select(col("Country"),col("State"),col("Country_code"),col("region"),col("customer_id"),col("Phone"),col("Source_file"),col("Ingested_at")).write.format('delta').mode('overwrite').saveAsTable(f"{catalog}.gold.gld_customers_dim")

###***Date***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

gld_dates=spark.read.table(f"{catalog}.silver.slv_dates")

gld_dates=gld_dates.withColumn('dateid',date_format(col('date'),'yyyyMMdd').cast(IntegerType()))\
    .withColumn('month_name',date_format(col('date'),'MMMM'))\
    .withColumn('is_weekend', when(col('day_name').isin(['Saturday','Sunday']),1).otherwise(0))

gld_dates.select(col('dateid'),col('date'),col('month_name'),col('year'),col('quarter'),col('Week'),col('day_name'),col('is_weekend'),col("Source_file"),col("Ingested_at")).write.format('delta').mode('overwrite').saveAsTable(f"{catalog}.gold.gld_dates_dim")